# Tutorial: Analysis of CASSCF Solutions

---

This tutorial shows how to interpret the results of a CASSCF calculation using different types of orbitals.
This tutorial assumes that you are already familiar with CASSCF claculations in Forte2.

### 💻 Importing the relevant modules from Forte2

To begin we will import a few components from Forte2:

In [ ]:
from pathlib import Path
import numpy as np

HAVE_MPL = True
try:
    import matplotlib.pyplot as plt
except ImportError as e:
    print(f"You likely need to install matplotlib: see original error message: {e}")
    HAVE_MPL = False

from forte2 import MCOptimizer, RHF, CISolver, State, System, write_orbital_cubes
from forte2.base_classes.params import DavidsonLiuParams

## 📚 Invariance of the CASSCF wavefunction with respect to orbital rotations

The CASSCF wavefunction is the sum of all the determinants in the active space ($| \Phi_I \rangle$) weighted by their corresponding CI coefficients ($c_I$):

\begin{equation}
| \Psi_\text{CASSCF} \rangle = \sum_I c_I | \Phi_I \rangle.
\end{equation}

In this expression, the Slater determinants are antisymmetrized products of spinorbitals ($\phi_i$), which we write as:

\begin{equation}
| \Phi_I \rangle = | \phi_{i_1} \phi_{i_2} \cdots \phi_{i_N} \rangle
\end{equation}

An important property of the CASSCF wavefunction is that it can be expressed in different orbital bases related by unitary transformations.
If we define a new set of spinorbitals ($\phi'_i$) as a linear combination of the original spinorbitals ($\phi_i$):

\begin{equation}
| \phi'_i \rangle = \sum_j U_{ij} | \phi_j \rangle,
\end{equation}

then we can defined a new set of Slater determinants ($| \Phi'_I \rangle$) in terms of the new spinorbitals:

\begin{equation}
| \Phi'_I \rangle = | \phi'_{i_1} \phi'_{i_2} \cdots \phi'_{i_N} \rangle.
\end{equation}

The CASSCF wavefunction can be expressed in terms of the new Slater determinants as:

\begin{equation}
| \Psi_\text{CASSCF} \rangle = \sum_I c'_I | \Phi'_I \rangle,
\end{equation}

where the new CI coefficients ($c'_I$) are related to the original ones by the unitary transformation.

This property of the CASSCF wavefunction implies that the orbitals are not uniquely defined.
**In order for an analysis of the CASSCF wavefunction to be well defined and reproducible**, we need to choose a specific orbital basis to represent the wavefunction.
Different options are available, and this tutorial will explore those implemented in Forte2.

## 📚 Semicanonical and natural orbitals

In this section we discuss two common methods for defining a consistent set of CASSCF orbitals:
1. **Semicanonical orbitals** are obtained by diagonalizing the active block of the generalized Fock matrix:
\begin{equation}
f_{pq} = h_{pq} + \sum_{rs} \langle pr||qs \rangle \gamma_{rs},
\end{equation}
where $h_{pq}$ is the one-electron Hamiltonian, $\langle pr||qs \rangle$ are the antisymmetrized two-electron integrals, and $\gamma_{rs}$ is the one-particle density matrix of the CASSCF wavefunction:
\begin{equation}
\gamma_{pq} = \langle \Psi_\text{CASSCF} | a^\dagger_p a_q | \Psi_\text{CASSCF} \rangle.
\end{equation}
Starting from general CASSCF orbitals, one forms the active block of the generalized Fock matrix and solves the eigenvalue problem:
\begin{equation}
\mathbf{f}^{\mathbb{A}} \mathbf{U}^{\mathbb{A}} = \mathbf{U}^{\mathbb{A}} \mathbf{\epsilon}^{\mathbb{A}}.
\end{equation}
The resulting eigenvectors $\mathbf{U}_{\mathbb{A}}$ determine the orbital transformation to obtain the semicanonical orbitals, while the corresponding eigenvalues $\mathbf{\epsilon}_{\mathbb{A}}$ are the **semicanonical orbital energies**.

2. **Natural orbitals**, which are obtained by diagonalizing the one-body density matrix of the CASSCF wavefunction in the active space:
\begin{equation}
\boldsymbol{\gamma}_1^{\mathbb{A}} \mathbf{U}^{\mathbb{A}} = \mathbf{U}^{\mathbb{A}} \mathbf{n}^{\mathbb{A}}.
\end{equation}
The eigenvalues $\mathbf{n}^{\mathbb{A}}$ are the **natural occupation numbers**, which are the eigenvalues of the one-particle density matrix in the active space.

In Forte2, **the default setting for CASSCF is to use semicanonical orbitals**. The code additionally uses semicanonical core and virtual orbitals, which are obtained by diagonalizing the core and virtual blocks of the generalized Fock matrix.

## 💻 Example: CO full-valence CAS(10,8)

To demonstrate the difference between semicanonical and natural orbitals, we run a full-valence CASSCF calculation on CO [CAS(10,8)].
The C and O 1s-like orbitals are kept doubly occupied.

### Semicanonical orbitals (default for CASSCF)

The following code block sets up a CASSCF calculation using the default value of the `final_orbitals` parameter, which is set to `semicanonical`. 

In [ ]:
xyz = f"""
C  0.0  0.0  0.0
O  0.0  0.0  2.5
"""

system = System(
    xyz=xyz,
    basis_set="cc-pVDZ",
    cholesky_tei=True,
    symmetry=False,
)
rhf = RHF(charge=0, e_tol=1.0e-12)(system)

singlet = State(system=system, multiplicity=1, ms=0.0)

cas_solver = CISolver(
    states=singlet,
    core_orbitals=2,
    active_orbitals=8,
)

casscf_default = MCOptimizer(cas_solver, e_tol=1.0e-12)(rhf)

casscf_default.run()

When expressed in the semicanonical orbital basis, the CASSCF wavefunction contains many leading determinants with significant contributions to the wavefunction.
This can be seen in the following table printed out in the output, which shows the top determinants in the CASSCF wavefunction along with their CI coefficients.
```md
Top determinants:
===========================================================================
Contrib.  #1           #2           #3           #4           #5           
---------------------------------------------------------------------------
Root 0    |222b20a0>   |222a20b0>   |2222ba00>   |2222ab00>   |222aabb0>   
          -0.287714    -0.287714    +0.287705    +0.287705    +0.233169    
===========================================================================
```

### Details: Helper functions

A helper function is defined in the next code block and used in the rest of the notebook to plot the active block of the Fock matrix and the one-particle density matrix in the active space.

In [ ]:
def plot_fock_and_rdm(mc):
    if not HAVE_MPL:
        print("matplotlib is not available, skipping plotting")
        return

    # compute the 1-RDM
    g1 = mc.make_sf_1rdm(0)

    # compute the Fock matrix in the MO basis
    fock_builder = mc.system.fock_builder
    mo_space = mc.mo_space
    C_contig = mc.mos.C[0][:, mo_space.orig_to_contig]
    C_docc, C_act = C_contig[:, mo_space.docc], C_contig[:, mo_space.actv]
    fock_ao = fock_builder.build_generalized_fock(
        C_core=C_docc,
        C_act=C_act,
        g1=g1,
    )
    f_mo = C_contig.conj().T @ fock_ao @ C_contig    
    f_act = f_mo[mo_space.actv][:, mo_space.actv]
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    fock_range = np.max(np.abs(f_act.real))
    im0 = axes[0].imshow(f_act.real, cmap='seismic', vmin=-fock_range, vmax=fock_range)
    axes[0].set_title('Fock Matrix in MO Basis')
    axes[0].set_xlabel('Active orbitals')
    axes[0].set_ylabel('Active orbitals')
    axes[0].set_xticks(range(f_act.shape[0]))
    axes[0].set_yticks(range(f_act.shape[0]))
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(g1, cmap='seismic', vmin=-2.0, vmax=2.0)
    axes[1].set_title('1-RDM in MO Basis')
    axes[1].set_xlabel('Active orbitals')
    axes[1].set_ylabel('Active orbitals')
    axes[1].set_xticks(range(g1.shape[0]))
    axes[1].set_yticks(range(g1.shape[0]))
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    fig.tight_layout()
    plt.show()

To verify that the final orbitals are semicanonical, we can evaluate the active block of the Fock matrix. The function below will use the information contained in a MCOptimizer object to evaluate the Fock matrix in the final CASSCF orbitals and return the active block of the Fock matrix. The function will also compute the one-particle reduced density matrix (1-RDM) in the final CASSCF orbitals basis. These two quantities are then plotted for comparison.

In [ ]:
plot_fock_and_rdm(casscf_default)

### Natural orbitals

Next, we examine the natural orbitals.
In the following input, we change the default value of the `final_orbitals` argument to `"natural"` and rerun the CASSCF calculation. The resulting 1-RDM is then plotted along with the active block of the Fock matrix.

In [ ]:
system = System(
    xyz=xyz,
    basis_set="cc-pVDZ",
    cholesky_tei=True,
    symmetry=False,
)
rhf = RHF(charge=0, e_tol=1.0e-12)(system)

singlet = State(system=system, multiplicity=1, ms=0.0)

cas_solver = CISolver(
    states=singlet,
    core_orbitals=2,
    active_orbitals=8,
    davidson_liu_params=DavidsonLiuParams(ndets_per_guess=100, maxiter=100)
)
casscf_nos = MCOptimizer(
    cas_solver,
    final_orbitals="natural",
)(rhf)
casscf_nos.run()

plot_fock_and_rdm(casscf_nos)

```md
Top determinants:
===========================================================================
Contrib.  #1           #2           #3           #4           #5           
---------------------------------------------------------------------------
Root 0    |22222000>   |2222ba00>   |2222ab00>   |222b20a0>   |222a20b0>   
          -0.511926    -0.223301    -0.223301    -0.223278    -0.223278    
===========================================================================
```

## Atomic-like localized orbitals

Forte2 also provides two additional options for final orbitals: intrinsic bond orbitals (IBOs) and IBOs with atomic-like character (IBO-atomic). These IBO orbitals are obtained by performing a unitary transformation of the semicanonical orbitals to maximize the localization of the orbitals in real space.

This procedure consists of the following steps:
1. Compute the normalized population of the $i$-th IBO on atom $A$ using the following formula:
\begin{equation}
    p_{Ai} = \sum_{\mu \in A} |\langle \mathrm{IAO}_\mu | \mathrm{IBO}_i \rangle|^2
\end{equation}
We consider an IBO to be important for atom $A$ if its population on that atom is greater than a threshold value ($\tau$, default: 0.9). This gives us a set of important IBOs for each atom, which we denote as $\mathcal{B}_A = \{ i | p_{Ai} > \tau \}$. Let's denote the size of this set $k$.

2. From the $k$ important IBOs relevant to an atom, we compute the population of the atomic orbitals relevant to the important IBOs using the following formula:
\begin{equation}
    p_{A\mu} = \sum_{i \in \mathcal{B}_A} |\langle \mathrm{IAO}_\mu | \mathrm{IBO}_i \rangle|^2
\end{equation}
We then select $k$ IAOs with the largest $p_{A\mu}$ values. This gives us a set of selected atomic orbitals for each atom, which we denote as $\mathcal{A}_A$.

3. Form the $k \times k$ submatrix $M_{\mu i} = \langle \mathrm{IAO}_\mu | \mathrm{IBO}_i \rangle$ with $\mathrm{IAO}_\mu \in \mathcal{A}_A$ and $\mathrm{IBO}_i \in \mathcal{B}_A$. If the minimum singular value of $M$ is smaller than $\tau$ we skip the alignment for this atom.

4. Next, we perform a Procrustes rotation of the important IBOs using the matrix $M$.

The following code runs the CASSCF calculation with the final orbitals set to `"ibo_atomic"`.

In [ ]:
system = System(
    xyz=xyz,
    basis_set="cc-pVDZ",
    cholesky_tei=True,
    symmetry=False,
)
rhf = RHF(charge=0, e_tol=1.0e-12)(system)

singlet = State(system=system, multiplicity=1, ms=0.0)

cas_solver = CISolver(
    states=singlet,
    core_orbitals=2,
    active_orbitals=8,
    davidson_liu_params=DavidsonLiuParams(ndets_per_guess=100, maxiter=100)
)
casscf_ibo_atomic = MCOptimizer(
    cas_solver,
    final_orbitals="ibo_atomic",
)(rhf)
casscf_ibo_atomic.run()

plot_fock_and_rdm(casscf_ibo_atomic)

To understand what this procedure accomplishes, we first consider the composition of the final active orbitals:
```md
AO Composition of active MOs:
# MO  (AO) label : coeff                      
2     C1 3s (2): +0.5716        C1 2s (1): +0.5101        O1 2pz (18): +0.0260      O1 3pz (21): +0.0208      C1 1s (0): +0.0175       
3     C1 2py (3): +0.6720       C1 3py (6): +0.4636       O1 2py (17): -0.0363      O1 3py (20): +0.0216      C1 3dyz (10): +0.0030    
4     C1 2pz (4): +0.6764       C1 3pz (7): +0.4679       O1 2pz (18): +0.0685      O1 3s (16): -0.0451       C1 3s (2): +0.0201       
5     C1 2px (5): +0.6720       C1 3px (8): +0.4636       O1 2px (19): -0.0363      O1 3px (22): +0.0216      C1 3dxz (12): +0.0030    
6     O1 3s (16): +0.5740       O1 2s (15): +0.5110       C1 2pz (4): -0.0349       C1 3pz (7): -0.0203       O1 1s (14): +0.0201      
7     O1 2py (17): +0.6914      O1 3py (20): +0.4541      C1 2py (3): -0.0212       C1 3py (6): +0.0070       C1 3dyz (10): +0.0037    
8     O1 2pz (18): +0.6970      O1 3pz (21): +0.4557      C1 2pz (4): +0.0455       C1 3s (2): +0.0224        C1 3pz (7): +0.0186      
9     O1 2px (19): +0.6914      O1 3px (22): +0.4541      C1 2px (5): -0.0212       C1 3px (8): +0.0070       C1 3dxz (12): +0.0037    
```
We can see that these orbitals are localized on the C and O atoms, and are dominated by C1(2/3s), C1(2py), C1(2pz), C1(2px), O1(2/3s), O1(2py), O1(2pz), and O1(2px) atomic orbitals. This ordering is produced intentionally by the IBO-atomic procedure.

In this basis, the CASSCF wavefunction can be expressed in terms of determinants where the occupied orbitals have well defined atomic character. For example, the table below show that the largest contributions to the CASSCF wavefunction is a configuration of the form:
\begin{equation}
| (2s^\mathrm{C})^2 (2p_{y}^\mathrm{C})^1 (2p_x^\mathrm{C})^1
(2s^\mathrm{O})^2 (2p_{y}^\mathrm{O})^1 (2p_z^\mathrm{O})^2 (2p_x^\mathrm{O})^1  \rangle
\end{equation}

```md
Top determinants:
===========================================================================
Contrib.  #1           #2           #3           #4           #5           
---------------------------------------------------------------------------
Root 0    |2a0a2b2b>   |2b0b2a2a>   |2aa02bb2>   |2bb02aa2>   |20bb22aa>   
          -0.324451    -0.324451    -0.304974    -0.304974    -0.304964    
===========================================================================
```

## Cube files for all orbitals

The following code block generates cube files for all orbitals in the calculation.

In [ ]:
for label, type in [("semicanonical", casscf_default), ("natural", casscf_nos), ("ibo_atomic", casscf_ibo_atomic)]:
    print(f"Writing orbital cubes for {label} orbitals...")
    write_orbital_cubes(
        system=system,
        C=type.mos.C[0],
        indices=list(range(10)),
        filepath=Path(f"co_cubes/{label}"),
    )